# Automated Factsheet Extraction Pipeline

This notebook provides a **single-run** extraction pipeline for extracting holdings data from multiple PDF factsheets.

## 📋 Usage Instructions

1. **Place your PDF factsheets** in the `factsheet_archive/` directory
2. **Run all cells** (Cell → Run All, or Shift+Enter for each cell)
3. **Output files** will be saved in `output/` directory (CSV and JSON)

## 📊 What This Notebook Does

- Extracts portfolio holdings from PDF factsheets
- Normalizes data across different AMC formats
- Validates AUM percentages
- Exports structured CSV and JSON files
- Generates processing summary and logs


In [ ]:
# Import required libraries
import sys
from pathlib import Path
import pandas as pd
from datetime import datetime
import logging

# Add parent directory to path for imports
sys.path.insert(0, str(Path().resolve().parent))

# Import project modules
from extractors.pdf_extractor import PDFExtractor
from processors.data_normalizer import DataNormalizer, DataAggregator
from processors.metadata_extractor import MetadataExtractor
from config import ARCHIVE_DIR, OUTPUT_DIR, MAPPINGS_DIR

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ All imports successful!")
print(f"📁 Archive directory: {ARCHIVE_DIR}")
print(f"📁 Output directory: {OUTPUT_DIR}")


In [ ]:
# Initialize data processors
normalizer = DataNormalizer(
    sector_mapping_path=str(MAPPINGS_DIR / "sector_mapping.json"),
    isin_mapping_path=str(MAPPINGS_DIR / "isin_mapping.json")
)
metadata_extractor = MetadataExtractor()

# Find all PDF files in archive directory
pdf_files = list(ARCHIVE_DIR.glob("*.pdf"))

if not pdf_files:
    print(f"⚠️  No PDF files found in {ARCHIVE_DIR}")
    print("Please add PDF factsheets to the factsheet_archive/ directory")
else:
    print(f"✅ Found {len(pdf_files)} PDF file(s) to process:")
    for pdf_file in pdf_files:
        print(f"   - {pdf_file.name}")


In [ ]:
# Process all PDFs and extract holdings
all_holdings = []
processing_log = []

print("\n" + "="*60)
print("STARTING EXTRACTION PROCESS")
print("="*60)

for idx, pdf_file in enumerate(pdf_files, 1):
    try:
        print(f"\n[{idx}/{len(pdf_files)}] Processing: {pdf_file.name}")
        logger.info(f"Processing: {pdf_file.name}")
        
        # Step 1: Extract metadata from filename and PDF content
        metadata = metadata_extractor.extract_from_filename(pdf_file.name)
        content_metadata = metadata_extractor.extract_from_content(str(pdf_file))
        metadata.update({k: v for k, v in content_metadata.items() if v})
        
        # Step 2: Extract holdings from PDF
        extractor = PDFExtractor(str(pdf_file))
        holdings = extractor.extract()
        
        if holdings:
            # Step 3: Normalize the extracted data
            normalized_holdings = normalizer.normalize(holdings, metadata)
            all_holdings.extend(normalized_holdings)
            
            # Step 4: Validate AUM percentages
            validation = normalizer.validate_aum(normalized_holdings)
            
            processing_log.append({
                'file': pdf_file.name,
                'status': 'success',
                'holdings_count': len(normalized_holdings),
                'total_percentage': validation.get('total_percentage', 0),
                'is_valid': validation.get('is_valid', False),
                'warnings': '; '.join(validation.get('warnings', []))
            })
            
            print(f"   ✅ Extracted {len(normalized_holdings)} holdings")
            if validation.get('total_percentage', 0) > 0:
                print(f"   📊 Total AUM: {validation.get('total_percentage', 0):.2f}%")
        else:
            processing_log.append({
                'file': pdf_file.name,
                'status': 'failed',
                'holdings_count': 0,
                'total_percentage': 0,
                'is_valid': False,
                'warnings': 'No holdings extracted - may be a summary factsheet'
            })
            print(f"   ❌ Failed to extract holdings")
            logger.warning(f"✗ Failed to extract from {pdf_file.name}")
    
    except Exception as e:
        logger.error(f"Error processing {pdf_file.name}: {str(e)}")
        processing_log.append({
            'file': pdf_file.name,
            'status': 'error',
            'holdings_count': 0,
            'total_percentage': 0,
            'is_valid': False,
            'warnings': f'Error: {str(e)}'
        })
        print(f"   ❌ Error: {str(e)}")

print("\n" + "="*60)
print(f"EXTRACTION COMPLETE: {len(all_holdings)} total holdings extracted")
print("="*60)


In [ ]:
# Create DataFrame and export to CSV/JSON
if all_holdings:
    df = pd.DataFrame(all_holdings)
    
    # Generate timestamp for output files
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = OUTPUT_DIR / f"holdings_{timestamp}.csv"
    json_path = OUTPUT_DIR / f"holdings_{timestamp}.json"
    
    # Export to CSV and JSON
    DataAggregator.export_to_csv(df, str(csv_path))
    DataAggregator.export_to_json(df, str(json_path))
    
    print("\n" + "="*60)
    print("EXPORT COMPLETE")
    print("="*60)
    print(f"✅ CSV exported to: {csv_path}")
    print(f"✅ JSON exported to: {json_path}")
    
    # Display summary statistics
    print("\n" + "="*60)
    print("EXTRACTION SUMMARY")
    print("="*60)
    print(f"Total Holdings: {len(df):,}")
    print(f"Unique Securities: {df['security_name'].nunique():,}")
    print(f"AMCs: {df['amc'].nunique()}")
    print(f"Funds: {df['fund_name'].nunique()}")
    
    if 'date' in df.columns and df['date'].notna().any():
        print(f"Date Range: {df['date'].min()} to {df['date'].max()}")
    
    if 'sector' in df.columns:
        print(f"\nTop Sectors by Holdings Count:")
        top_sectors = df['sector'].value_counts().head(5)
        for sector, count in top_sectors.items():
            print(f"  - {sector}: {count} holdings")
    
    print("="*60)
    
    # Display first few rows
    print("\n📊 First 10 Holdings:")
    display(df.head(10))
    
    # Display data types and info
    print("\n📋 Data Info:")
    print(df.info())
    
else:
    print("\n❌ No holdings extracted from any files")
    print("Please check:")
    print("  1. PDF files are complete factsheets (not summaries)")
    print("  2. PDFs contain 'Portfolio Holdings' tables")
    print("  3. OpenAI API key is set in .env file (for LLM extraction)")
    logger.error("No holdings extracted from any files")


In [ ]:
# Save processing log for review
if processing_log:
    log_df = pd.DataFrame(processing_log)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = OUTPUT_DIR / f"processing_log_{timestamp}.csv"
    log_df.to_csv(log_path, index=False)
    
    print("\n" + "="*60)
    print("PROCESSING LOG")
    print("="*60)
    print(f"✅ Log saved to: {log_path}")
    print("\nProcessing Summary:")
    display(log_df)
    
    # Summary statistics
    success_count = len(log_df[log_df['status'] == 'success'])
    failed_count = len(log_df[log_df['status'] == 'failed'])
    error_count = len(log_df[log_df['status'] == 'error'])
    
    print(f"\n📊 Processing Statistics:")
    print(f"   ✅ Successful: {success_count}")
    print(f"   ❌ Failed: {failed_count}")
    print(f"   ⚠️  Errors: {error_count}")
    print("="*60)
else:
    print("No processing log to save")
